[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/7_SmallNetworks.ipynb)

# Set 7 — Small Networks: STM & WTA (LEGO–Colab)
**Author:** Neural Engineering Laboratory, University of Missouri-
Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

---

# **Variable Key & Roadmap**

### **The "Levers" (Network Parameters)**
These parameters control the biophysical behavior of the circuits you will build below.

| **Variable** | **Definition** | **Unit** | **Instructional Role** |
| :--- | :--- | :--- | :--- |
| **`weight_EE`** | Excitatory to Excitatory strength | μS | Controls "Reverberation" / Persistence duration in STM. |
| **`E2I_weight`** | Excitatory to Inhibitory strength | μS | Controls how easily a Principal cell recruits its partner interneuron. |
| **`II_weight`** | Inhibitory to Excitatory strength | μS | Controls competitive suppression (The "Veto" in Winner-Take-All). |
| **`syn_delay`** | Synaptic delay | ms | Affects the timing and stability of decisions. |
| **`g_pas`** | Passive Leak Conductance | S/cm² | Determines the "forgetfulness" or decay rate of the membrane. |

---

### **The Roadmap**
This module explores how small groups of neurons work together to perform basic cognitive functions.
* **[F0 Starter](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=QwiJS79RIitS):** Initializing the multi-cell "Engine" and helper builders.
* **[F1 Excitatory STM](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=v7ZlK6x8Yp6P):** Investigating if a brief pulse can trigger a memory that outlasts the stimulus.
* **[W — Winner-Take-All](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=874e5088):** Using mutual inhibition to force the network to "choose" a winner.
* **[Model Validation](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=95cbf906):** Verifying the synaptic logic with a simplified 2-cell test.
* **[Practice & Reflection](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=aca529ba):** Testing the limits of decision-making and metabolic costs.
* **Capstone Challenge:** Vibe-coding a cross-simulator translation of your WTA circuit (NEURON → Brian2).
* **Central Pattern Generators:** A third network motif — reciprocal inhibition plus slow escape, producing rhythm; scales up to gait networks (walk, trot, pace, gallop).
* **Homework + Knowledge Bank:** Spanning STM, WTA, and CPGs together.
* **🏆 Final Capstone:** Design a real half-center oscillator from scratch — pure vibe-coding, synthesizing Sets 5, 6, and 7.

---
### A Note on This Notebook
**This completed, filled-in notebook is the lab deliverable.** Add your own cells as you go — the **Capstone Challenge** below already asks you to translate your circuit into a new tool; feel free to extend it further.

In [2]:
#@title F0: Initialize Environment (Realistic HH) { display-mode: "form" }
#@markdown This cell installs NEURON and defines the core 'Engine' functions.

import os
import sys

# 1. Install NEURON if not present
try:
    import neuron
    from neuron import h, gui
    print("NEURON is already installed.")
except ImportError:
    print("NEURON not found. Installing...")
    os.system('pip install neuron')
    from neuron import h, gui
    print("NEURON installation complete.")

import matplotlib.pyplot as plt
import numpy as np

# 2. Updated Builder Function: Handles multiple cell types
def build_population(n, cell_type='exc', g_na=0.12, g_k=0.036):
    """
    Creates a list of n Hodgkin-Huxley sections.
    cell_type: 'exc' for Excitatory (Principal) or 'inh' for Inhibitory.
    """
    pop = []
    for i in range(n):
        sec = h.Section(name=f'{cell_type}_{i}')
        sec.L = 20

        sec.diam = 20

        # INSERT MECHANISMS
        sec.insert('hh')   # Active channels (na, k, gl)
        sec.insert('pas')  # Passive leak mechanism

        # Apply custom channel conductances if present
        if h.ismembrane('hh', sec=sec):
            sec.gnabar_hh = g_na
            sec.gkbar_hh = g_k

        # Differentiation: Inhibitory cells often have faster/leakier membranes
        if cell_type == 'inh':
            sec.g_pas = 0.0002  # Leak conductance
            sec.e_pas = -65     # Leak reversal
        else:
            sec.g_pas = 0.0001
            sec.e_pas = -65

        pop.append(sec)
    return pop


# 3. Helper: Tonic Current Clamp
def tonic_iclamp(section, amp=0.1, delay=5.0, duration=1e9):
    stim = h.IClamp(section(0.5))
    stim.delay = delay
    stim.dur = duration
    stim.amp = amp
    return stim

def brief_iclamp(section, amp=0.3, delay=20.0, dur=20.0):
    stim = h.IClamp(section(0.5))
    stim.delay = delay
    stim.dur = dur
    stim.amp = amp
    return stim

# 4. Helper: Recording Setup
def mk_rec(section):
    v = h.Vector().record(section(0.5)._ref_v)
    return v

# 5. Helper: Plotting Logic
def plot_traces(t, voltages, labels, title="Neural Activity"):
    plt.figure(figsize=(10, 5))
    for v, l in zip(voltages, labels):
        plt.plot(t, v, label=l)
    plt.xlabel('Time (ms)')
    plt.ylabel('Membrane Potential (mV)')
    plt.title(title)
    plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
    plt.grid(True, alpha=0.3)
    plt.show()

print("F0 Engine initialized and ready.")

NEURON is already installed.
F0 Engine initialized and ready.


In [ ]:
# Helper: connect_all_to_all — create all-to-all synapses between cells
def connect_all_to_all(cells, tau=5.0, weight=0.01, delay=0.5, excitatory=True):
    """
    Connect every cell i -> j (i != j) using an ExpSyn on the target cell.
    Returns (syns, netcons).
    Parameters:
        cells : list of Sections or objects with callable indexing (e.g., cell(0.5))
        tau : synaptic decay time constant (ms)
        weight : synaptic weight for NetCon (units depend on mechanism)
        delay : conduction delay (ms)
        excitatory : if False, sets synapse reversal to inhibitory (-80 mV)"""
    syns = []
    ncs = []
    for i, src in enumerate(cells):
        for j, tgt in enumerate(cells):
            if i == j:
                continue
            # attach an ExpSyn to the target at 0.5
            syn = h.ExpSyn(tgt(0.5))
            # set reversal based on excitatory/inhibitory flag
            syn.e = 0.0 if excitatory else -80.0
            # many ExpSyn implementations expose 'tau' attribute
            try:
                syn.tau = tau
            except Exception:
                pass
            # create a NetCon from source voltage to this synapse
            nc = h.NetCon(src(0.5)._ref_v, syn, sec=src)
            nc.weight[0] = weight
            nc.delay = delay
            syns.append(syn)
            ncs.append(nc)
    return syns, ncs

# Backwards-compatible alias (in case other cells expect this name)
connect_all_to_all_eps = connect_all_to_all
print("Added helper: connect_all_to_all")

Added helper: connect_all_to_all


: 

### **F1 — STM (excitatory all-to-all)**

**Idea**: A brief pulse to one excitatory unit enters a recurrent $E \to E$ network. Sufficient weight and excitatory $\tau$ produce post-stimulus persistence, a hallmark of Short-Term Memory (STM).

**Activity — Ask AI:** *"If you connect several excitatory neurons to each other in a loop (each one excites the others), what happens after a single brief pulse to just one of them? Could the network keep firing even after the pulse ends, with no external input? Explain why or why not."* Paste the response.

*Your Answer Here — does this match what 'persistence' means?*

**What to change**  
*   **Weight ($E \to E$):** Approximately 0.004–0.010.
*   **$\tau$ (E synaptic decay):** 1–5 ms.
*   **Delay ($E \to E$):** 0.5–2 ms.
*   **Stimulus:** Pulse amplitude and duration in `brief_iclamp`.

**Sanity checks**  
1.  Observe if spiking continues after the external pulse ends.
2.  Report the **persistence duration** (time from pulse offset to the last recorded spike).

**Predict → verify**  
*   **Persistence:** Increasing weight or $\tau$ should lengthen persistence until it reaches a saturation regime.
*   **Stability:** Excessive values may yield synchronized or runaway spiking.

**Exercises**  
1.  Find a (weight, $\tau$) pair that yields **50–150 ms** of persistence; capture the raster/trace and record the duration.
2.  Double the membrane leak ($g_{pas}$) and re-measure; explain the change using RC integrator logic.
3.  Identify a parameter set that fails to sustain activity; state which specific lever is currently limiting the network.

---



In [ ]:
# @title F1: STM Challenge - Pulse Ignition { display-mode: "form" }
from ipywidgets import interactive_output, HBox, VBox, FloatSlider, Layout, Accordion, HTML, Checkbox
import matplotlib.pyplot as plt
import numpy as np
from neuron import h

def simulate_f1_toggle(w_ee, tau_e, delay, g_na, g_k, g_pas, stim_amp, network_mode):
    h('forall delete_section()')

    # Toggle between 1-cell (no loop) and 2-cell (ping-pong loop)
    N = 2 if network_mode else 1
    cells = build_population(N, g_na=g_na, g_k=g_k)

    v_rest = -65.0
    for c in cells:
        sections = c.all if hasattr(c, 'all') else [c]
        for sec in sections:
            if h.ismembrane('pas', sec=sec):
                sec.g_pas = g_pas
                sec.e_pas = v_rest
            if h.ismembrane('hh', sec=sec):
                sec.gnabar_hh = g_na
                sec.gkbar_hh = g_k

    # Only connect if we have 2 cells
    if network_mode and N > 1:
        syns, ncs = connect_all_to_all(cells, tau=tau_e, weight=w_ee, delay=delay)

    # Ignition Pulse
    stim_in = brief_iclamp(cells[0], delay=20, dur=20, amp=stim_amp)

    t = h.Vector().record(h._ref_t)
    recs = [mk_rec(c) for c in cells]

    def custom_init():
        h.v_init = v_rest
        for sec in h.allsec():
            sec.v = v_rest
            if h.ismembrane('hh', sec=sec):
                sec.m_hh = 0.0529; sec.h_hh = 0.5961; sec.n_hh = 0.3177

    finit = h.FInitializeHandler(custom_init)
    h.finitialize(v_rest)
    h.continuerun(250)

    plt.figure(figsize=(10, 4.8))
    plt.plot(t, np.array(recs[0]), label='Cell 0 (Input Recipient)', color='#1f77b4', lw=2)
    if network_mode:
        plt.plot(t, np.array(recs[1]) - 2, label='Cell 1 (Loop Partner)', color='#ff7f0e', lw=2, alpha=0.7)

    plt.title("F1: STM Challenge | " + ("2-Cell Recurrent Loop" if network_mode else "Single Cell (No Loop)"))
    plt.axvspan(20, 40, color='green', alpha=0.1, label='WRITE (Pulse)')
    plt.axhline(-20, color='red', linestyle='--', alpha=0.2, label='Threshold')

    plt.xlim(0, 250)
    plt.ylim(-85, 45); plt.ylabel('V (mV)'); plt.xlabel('Time (ms)')
    plt.legend(loc='upper right', fontsize='small'); plt.grid(alpha=0.2); plt.show()

# --- UI SETUP ---
key_content = HTML("""
<b>Variable Key:</b>
<ul>
  <li><b>Weight_EE:</b> Strength of the connection between the two cells.</li>
  <li><b>Network Mode:</b> When OFF, you see a single cell. When ON, the loop is closed.</li>
</ul>
""")
acc = Accordion(children=[key_content])
acc.set_title(0, 'Roadmap & Variable Key')
acc.selected_index = None

s_lay = Layout(width='140px', height='200px')
sa_s  = FloatSlider(value=0.35, min=0.0, max=0.8, step=0.01, description='Stim_Amp', orientation='vertical', layout=s_lay)
w_s   = FloatSlider(value=0.34, min=0.0, max=1.5, step=0.01, description='Weight_EE', orientation='vertical', layout=s_lay)
na_s  = FloatSlider(value=0.10, min=0.0, max=0.4, step=0.01, description='g_Na', orientation='vertical', layout=s_lay)
k_s   = FloatSlider(value=0.06, min=0.0, max=0.2, step=0.01, description='g_K', orientation='vertical', layout=s_lay)
pas_s = FloatSlider(value=0.0001, min=0.0, max=0.001, step=0.00001, description='g_pas', orientation='vertical', layout=s_lay, readout_format='.5f')
t_s   = FloatSlider(value=5.0, min=1.0, max=50.0, step=1.0, description='Tau_Syn', orientation='vertical', layout=s_lay)
d_s   = FloatSlider(value=19.1, min=0.1, max=30.0, step=0.1, description='Delay', orientation='vertical', layout=s_lay)

# Restore the 1-vs-2 cell toggle
net_check = Checkbox(value=False, description='Enable Network (2-Cell)')

params = {'w_ee': w_s, 'tau_e': t_s, 'delay': d_s, 'g_na': na_s,
          'g_k': k_s, 'g_pas': pas_s, 'stim_amp': sa_s, 'network_mode': net_check}

display(VBox([acc, net_check, HBox([sa_s, w_s, na_s, k_s, pas_s, t_s, d_s]),
              interactive_output(simulate_f1_toggle, params)]))

### 🚫 By-Hand Drill — No AI, No Code
Persistence in this network is really just RC-integrator logic from Set 3, applied to a loop instead of a single cell.

By hand: if $\tau_m = R_{in} \cdot C_m$ for a single cell, and doubling $g_{pas}$ halves $R_{in}$, explain in one sentence why doubling $g_{pas}$ should shorten persistence duration — without running anything.

Then, using the same logic: if increasing synaptic weight effectively adds MORE current keeping each cell above threshold, explain why there might be a weight value beyond which the network no longer just persists, but fires faster and faster (runaway).

*Your Answer Here (both explanations):*

# **W — Winner-Take-All (WTA) via Mutual Inhibition**

### **1. The Biological Concept**
In this module, we move from single-cell persistence to a network-level decision: **Reciprocal (Mutual) Inhibition**. This is a circuit motif found throughout the brain, from the spinal cord to the cortex.

In this architecture, each excitatory Principal cell has a "dedicated" inhibitory partner. When a Principal cell fires, it recruits its partner to silence all *other* competitors. This creates a "Hard WTA" state where the network must commit to a single "winner" based on the strongest input.

**Activity — Ask AI:** *"If you connect several neurons so that whichever one fires first strongly inhibits all the others, what happens when you give them all slightly different amounts of input current? Will more than one ever remain active? Explain the mechanism."* Paste the response.

*Your Answer Here — how is this fundamentally different from the STM network's recurrent excitation?*

### **2. The "Levers" (Network Parameters)**
| Variable | Role in this Module | Unit | Instructional Role |
| :--- | :--- | :--- | :--- |
| **`E2I_weight`** | E → I coupling | μS | How easily a cell recruits its "defensive" interneuron. |
| **`II_weight`** | I → E competition | μS | The strength of the "veto." High values lead to faster, crisper decisions. |
| **`Input Gradient`** | $\Delta$ Excitation | nA | The "unfairness" of the race. Represents signal contrast. |

### **3. Predict → Verify**
> **Scenario:** We are providing a linear gradient of current from **Cell 0 (0.08 nA)** to **Cell 4 (0.12 nA)**.
> * **Predict:** If you set the `II_weight` to `0`, what will the raster plot look like?
> * **Verify:** Run the simulation below. Does the cell with the highest excitation successfully silence the others? How long does the "race" last before a winner is declared?

### 🚫 By-Hand Drill — No AI, No Code
Reasoning through a real WTA scenario before touching any code.

Five cells receive these input currents: $I_1=0.3210$, $I_2=0.2200$, $I_3=0.2190$, $I_4=0.2210$, $I_5=0.2195$ mA.

By hand:
1. Which cell should win, based purely on "strongest input wins"?
2. Rank the remaining four cells by how close a "contest" they'd put up, from closest to least close.
3. If $I_3$ and $I_5$ were swapped, would your predicted winner change? Why or why not?

*Your Answer Here — then check your predictions once you run the actual simulation below.*

In [1]:
#@title W2: Mutual (I-I) Inhibition Architecture { display-mode: "form" }
#@markdown ### Network Configuration
N = 5
II_weight = 0.18 #@param {type:"slider", min:0, max:0.5, step:0.01}
E2I_weight = 0.01 #@param {type:"slider", min:0, max:0.05, step:0.001}
syn_delay = 0.5 #@param {type:"slider", min:0.1, max:5.0, step:0.1}

#@markdown ### Individual Excitation (Input Amps)
#@markdown Adjust these to decide which cell "wins" the race.
amp_0 = 0.08 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_1 = 0.18 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_2 = 0.03 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_3 = 0.01 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_4 = 0.01 #@param {type:"slider", min:0, max:0.2, step:0.01}

#@markdown ### Cell Biophysics
g_na = 0.12 #@param {type:"slider", min:0, max:0.5, step:0.01}
g_k = 0.036 #@param {type:"slider", min:0, max:0.1, step:0.005}

import numpy as np
from neuron import h
import matplotlib.pyplot as plt

# 1. BUILD
princ = build_population(N, cell_type='exc')
inh_cells = build_population(N, cell_type='inh')

for p in princ + inh_cells:
    p.gnabar_hh = g_na
    p.gkbar_hh = g_k

# 2. INPUTS: Using the individual sliders
amps = [amp_0, amp_1, amp_2, amp_3, amp_4]
ics = [tonic_iclamp(princ[i], amp=float(amps[i]), delay=5.0) for i in range(N)]

# 3. E -> I Trigger
for i in range(N):
    syn = h.ExpSyn(inh_cells[i](0.5))
    nc = h.NetCon(princ[i](0.5)._ref_v, syn, sec=princ[i])
    nc.threshold = 0.0
    nc.weight[0] = E2I_weight
    nc.delay = 0.5

# 4. I -> E Competition (The Veto)
for i in range(N):
    for j in range(N):
        if i != j:
            syn = h.ExpSyn(princ[j](0.5))
            syn.e = -80.0
            nc = h.NetCon(inh_cells[i](0.5)._ref_v, syn, sec=inh_cells[i])
            nc.threshold = 0.0
            nc.weight[0] = II_weight
            nc.delay = syn_delay

# --- RUN ---
t = h.Vector().record(h._ref_t)
recP = [mk_rec(p) for p in princ]
h.finitialize(-65)
h.continuerun(500)

# --- PLOT ---
plot_traces(np.array(t), [np.array(r) for r in recP], [f'P[{i}]' for i in range(N)],
            title=f'W2: WTA (II_weight={II_weight})')

Traceback (most recent call last):
  File "_pydevd_bundle/pydevd_cython.pyx", line 1078, in _pydevd_bundle.pydevd_cython.PyDBFrame.trace_dispatch
  File "_pydevd_bundle/pydevd_cython.pyx", line 297, in _pydevd_bundle.pydevd_cython.PyDBFrame.do_wait_suspend
  File "c:\Users\nairs\Anaconda3\envs\bmtk\lib\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 1976, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, suspend_type, from_this_thread, frames_tracker)
  File "c:\Users\nairs\Anaconda3\envs\bmtk\lib\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 2011, in _do_wait_suspend
    time.sleep(0.01)
KeyboardInterrupt


KeyboardInterrupt: 

### **Model Validation: Functional Connectivity Test**
Before analyzing the full 5-cell network, we validate the underlying synaptic logic. This test isolates two neurons to confirm that the inhibitory "veto" is functioning correctly.

* **The Goal:** Confirm that activity in **P[1]** (Trigger) successfully suppresses **P[0]** (Target).
* **Success Criteria:** A successful test is indicated by the target cell's membrane potential dropping below its resting level while the trigger cell is spiking.

In [ ]:
#@title Model Validation: Functional Connectivity Test { display-mode: "form" }
#@markdown This cell runs a "mini-simulation" to prove that P[1] can successfully silence P[0].

import numpy as np
from neuron import h
import matplotlib.pyplot as plt

# 1. Setup a minimal test (2 cells)
test_princ = build_population(2, cell_type='exc')
test_inh = build_population(2, cell_type='inh')

2

## **Synthesis So Far: Two Cognitive LEGOs**
*(A third motif — Central Pattern Generators — and a final capstone are still ahead.)*

We have now explored two fundamental circuit motifs:

1.  **Short-Term Memory (F1):** Driven by **Recurrent Excitation (E→E)**. It allows a transient signal to persist, acting as a "buffer" for information.
2.  **Winner-Take-All (W):** Driven by **Mutual Inhibition (I→I)**. It allows a network to pick the strongest signal and suppress noise, acting as a "decision-maker."

**Discussion Question for Class:**
*In a real brain, these two motifs are often combined. If you had a network that had BOTH E→E persistence and I→I competition, how would that change the way you make a decision compared to a network that only has competition?*

## **Capstone Challenge: Vibe-Coding a Cross-Simulator Translation (NEURON → Brian)**

*Note: unlike most exercises in this Set, this activity deliberately uses an AI assistant to generate code — a "vibe-coding" exercise. The goal isn't to learn Brian's syntax; it's to test whether you understand the WTA circuit well enough to direct, verify, and correct an AI's implementation of it in a tool you've never used.*

**Why this exercise exists:** NEURON and Brian are both real, widely used spiking-network simulators, but they think about a model very differently — NEURON builds up compartments and mechanisms procedurally; Brian describes a population's dynamics as a block of differential equations (a `NeuronGroup`) and connects populations with a `Synapses` object. If you can re-implement your Winner-Take-All circuit's *logic* in a completely different tool, that's strong evidence you understand the circuit itself, not just the NEURON code that happens to run it.

### Step 1 — Install Brian2
Brian2 isn't preinstalled in this Colab (only NEURON is). Run the cell below once.

In [ ]:
!pip install brian2 --quiet
import brian2
print('Brian2 version:', brian2.__version__)

### Step 2 — Describe Your Circuit to the AI (Don't Skip This)
Before asking for any code, get the AI to state the circuit logic back to you in plain language — if it can't do that correctly, its code won't be right either.

> **Suggested Prompt 1:** *"I have a NEURON-based network with 5 excitatory 'principal' cells, each with its own dedicated inhibitory partner cell. Each principal cell excites only its own inhibitory partner (E→I). Every inhibitory cell inhibits every principal cell except its own (mutual/reciprocal I→E inhibition). Each principal cell receives a different constant input current. Explain, in plain language and independent of NEURON syntax, what circuit-level computation this architecture performs and why."*

Paste the AI's response below. Check it against what you already know from the **W — Winner-Take-All** section above — does it correctly describe the "veto" logic?

*Your Answer Here:*

### Step 3 — Translate to Brian2
Now ask for the actual translation. To keep the translation tractable, we simplify each Hodgkin-Huxley cell to a **leaky integrate-and-fire (LIF)** neuron — the circuit *architecture* (E→I, mutual I→E) is what we're testing, not the ion-channel biophysics.

> **Suggested Prompt 2:** *"Using Brian2 (Python), implement a spiking neural network with 5 excitatory leaky integrate-and-fire neurons, each with one dedicated inhibitory LIF partner. Each excitatory neuron should provide a different constant input current, feed-forward excite only its own inhibitory partner, and each inhibitory neuron should inhibit every excitatory neuron except the one it's paired with. Use Brian2's NeuronGroup and Synapses objects, and include a SpikeMonitor so I can plot a raster of all 10 neurons."*

Paste the AI-generated Brian2 code below and run it.

In [ ]:
# Paste and run your AI-generated Brian2 code here.
# Tip: give each excitatory neuron a different constant input current (analogous to amp_0..amp_4
# in the NEURON version above) so you can compare directly to the WTA race you already ran.

### Step 4 — Verify, Don't Trust
Run your translated network and check it actually behaves like a Winner-Take-All circuit — the highest-input excitatory neuron should end up firing while the others are suppressed, just like in the NEURON version above.

> **Suggested Prompt 3 (if it doesn't work):** *"My Brian2 raster plot shows [describe what you actually see — e.g., 'all neurons still spiking' or 'no spikes at all']. Here is my code: [paste it]. What's likely wrong, given that I want mutual inhibition to eventually suppress all but the strongest-input neuron?"*

AI-generated code for an unfamiliar tool is often subtly wrong — a common failure here is inhibitory neurons accidentally connecting to themselves, or a missing reset/threshold on the LIF equations. Debugging this *is* the exercise.

### Reflection
1. **What had to change?** List at least two things that had to be simplified or restructured to move from your NEURON model to Brian2 (e.g., HH biophysics → LIF, procedural cell-building → declarative `NeuronGroup` equations).
2. **What stayed conceptually the same?** Despite the syntax being completely different, which parts of the *circuit logic* (the E→I / I→E wiring pattern, the competitive dynamic) transferred directly?
3. **Did you have to correct the AI?** If your first AI-generated attempt didn't produce a working Winner-Take-All pattern, what was wrong, and how did you know — what specifically in your own understanding of the circuit let you catch the mistake?

*Your Answer Here:*

---
## A Third Network Motif: Central Pattern Generators
You've now seen two circuit motifs: **recurrent excitation** (STM, persistence) and **mutual inhibition** (WTA, competition). There's a third, and it's responsible for one of the most common things brains do: producing **rhythm** — walking, breathing, swimming, chewing — without needing rhythmic input to drive it.

**Activity — Ask AI:** *"What is a central pattern generator (CPG) in neuroscience, and how is it different from a reflex? Specifically, what does it mean for a rhythmic behavior to be produced 'endogenously'?"* Paste the response.

*Your Answer Here:*

**The ladder of movement types**, from simplest to most complex: reflexes (involuntary, no threshold, e.g., knee-jerk) → fixed action patterns (involuntary, has a threshold, e.g., sneezing) → directed movements (voluntary, not repetitive, e.g., reaching) → **rhythmic motor patterns** (complex AND stereotyped AND repetitive — produced by CPGs).

### The Half-Center Oscillator: The Minimal CPG
The simplest possible CPG is just **two mutually inhibitory populations**, each driving its own output (e.g., motoneurons for a D-phase and a V-phase of a movement cycle) — this is the **half-center oscillator (HCO)**.

Here's the puzzle: if the two sides just inhibit each other, once one side wins, shouldn't it win forever — like WTA? What makes a half-center **oscillate** instead of settling into one permanent winner?

**Activity — Ask AI:** *"In a half-center oscillator, two neurons mutually inhibit each other, yet the network produces alternating rhythmic bursting rather than one side permanently silencing the other. What additional mechanism, beyond simple mutual inhibition, is required to make this happen? Consider what would let the inhibited side 'escape' from inhibition over time."* Paste the response.

*Your Answer Here — how is this different from what keeps a WTA winner permanently on top?*

**The key addition:** a slow process (e.g., adaptation, or a slow inward current like $I_h$) that gradually lets the inhibited side build up enough drive to escape and take over — flipping the winner, again and again.

### 🚫 By-Hand Drill — No AI, No Code
Reasoning through the escape mechanism before building anything.

Suppose Side A is currently firing (inhibiting Side B), and Side B is slowly building up an escape current.
1. If the escape current builds up VERY slowly, what happens to the oscillation period (long or short)?
2. If mutual inhibition strength is increased, does Side B need to build up MORE or LESS escape current before it can break free? What does that do to the period?
3. What would happen to the whole rhythm if you fully removed the escape mechanism, leaving only mutual inhibition?

*Your Answer Here (all three):*

### Vibe-Code a Minimal Half-Center Oscillator (Rate Model)
Let's build the simplest possible version — not biophysical cells yet, just two rate units with mutual inhibition and a slow escape variable — to see the rhythm emerge before building the full spiking version in the capstone.

> **Suggested prompt:** *"Write Python code simulating two mutually inhibitory rate units (firing rate r1, r2, each 0 to 1), where each unit also has a slow adaptation variable that builds up while it's active and decays while it's silent, and increased adaptation reduces that unit's own firing rate. Use Euler's method to simulate 2000 ms, plot r1 and r2 over time, and tune parameters until you see clean alternating bursts."*

Paste and run the code below.

In [ ]:
# Your vibe-coded half-center rate-model code here:


*Your Answer Here — what parameter did you have to tune to get clean alternation, and what happened when you set the escape/adaptation term to zero?*

### Scaling Up: Inter-Segmental Coordination and Gaits
One half-center oscillator controls one joint's flexion/extension cycle. But animals have many segments — and the *relative timing* between multiple HCOs determines the entire gait.

A single fish segment produces a **wave of activation** traveling down the body. Add left-right and front-back coordination, and you get every tetrapod gait: **walk**, **trot**, **pace**, and **gallop** — all from the same basic HCO building block, just phase-shifted differently between segments.

**Design Challenge 1 — Pacing:** Compare the coordination patterns of trot vs. pace (trot: diagonal leg pairs move together; pace: same-side leg pairs move together). Using your rate-model HCO as a building block, connect four coupled oscillators (one per leg) and find the phase relationships that produce a **pacing** gait. Suggested starting point: set all coupling weights equal, then adjust relative phase offsets.

*Your Answer Here — what phase relationship between the four HCOs did you land on for pacing?*

In [ ]:
# Your vibe-coded 4-HCO pacing network here:


**Design Challenge 2 — Gallop:** Gallop is the most asymmetric gait — nothing is simply "opposite" or "together." Extend your 4-HCO network to produce a galloping pattern.

*Your Answer Here — describe the phase relationships you needed, and how they differ from pacing's.*

**Design Challenge 3 — Breaking the Rhythm:** Take any working gait network above and weaken the mutual inhibition between left and right sides until the rhythm becomes unreliable or stops alternating cleanly.

*Your Answer Here — at what point does the rhythm break down, and does this match your by-hand prediction from earlier in this section?*

---
## Homework: This Is What Quizzes and Exams Will Look Like
- **By hand, no AI:** the persistence/RC-integrator reasoning (STM), the winner-prediction ranking (WTA), the escape-mechanism reasoning (CPG)
- **Design deliverables:** a stable STM persistence window, a clean WTA winner, a working half-center rate model, a 4-HCO pacing network, a 4-HCO gallop network
- **Diagnostic exercise:** breaking the CPG rhythm by weakening inhibition, and explaining why

---
## Knowledge Bank — Set 7: Small Networks (STM, WTA, and CPGs)

**This is the outcome of this module.** Quizzes, the midterm, and the final draw directly from these questions — know them well. For anything you're unsure about, use an AI assistant to research it, and add what you find as your own notes right in this notebook.

### A. Short-Term Memory (Recurrent Excitation)
1. What circuit motif underlies STM in this module, and why does it allow activity to outlast the triggering stimulus?
2. If you doubled the membrane leak conductance, would persistence get longer or shorter? Explain using RC-integrator logic.
3. What happens if excitatory weight is set too high — does the network still show simple persistence?

### B. Winner-Take-All (Mutual Inhibition)
4. What circuit motif underlies WTA, and how is it fundamentally different from STM's motif?
5. Given five input currents, how do you predict which cell wins without running a simulation?
6. What is "Soft-WTA," and why might a biological circuit prefer it over "Hard-WTA" in some situations?
7. Why is synaptic delay relevant to how "clean" a WTA decision looks?

### C. Central Pattern Generators and the Half-Center Oscillator
8. Define a central pattern generator. How is a CPG's output different from a reflex's?
9. Why doesn't simple mutual inhibition alone produce oscillation the way it does in WTA?
10. What additional mechanism is required for a half-center oscillator to alternate rather than settle on one permanent winner?
11. If the escape/adaptation mechanism builds up very slowly, what happens to the oscillation period?
12. How do multiple half-center oscillators, phase-shifted relative to each other, produce different gaits (walk, trot, pace, gallop)?
13. What happens to a gait network's rhythm if inter-side inhibition is weakened too much?

### D. Synthesis Across Motifs
14. Name a real-world task where a brain circuit might need both recurrent excitation (persistence) and mutual inhibition (competition) simultaneously.
15. If one inhibitory interneuron in a WTA circuit were removed, how would competition change for the remaining cells?
16. Why is a "Hard" WTA decision more metabolically expensive than a "Soft" one?
17. Across all three motifs (STM, WTA, CPG), what is the common role played by an interneuron?
18. Why does neither STM nor WTA, on their own, explain how a brain produces genuinely rhythmic behavior?

---
# 🏆 Final Capstone: Design a Real Half-Center Oscillator

Everything in this section is a synthesis challenge. This used to be a half-semester modeling project — building a two-cell half-center oscillator in NEURON, week by week. You're going to do the same thing in one sitting, because **you already have every piece you need**:

- **Set 5** taught you how to build single spiking, adapting, and bursting cells from voltage-gated conductances.
- **Set 6** taught you how to connect cells with inhibitory synapses, including precise timing control.
- **This Set** just taught you exactly why reciprocal inhibition alone isn't enough for oscillation — you need a slow escape mechanism.

The half-center oscillator is just: **two bursting or adapting cells, reciprocally connected by inhibitory synapses.** That's the whole design.

### 🚫 Step 1 — Predict First, No AI
Before building anything: sketch what you expect the membrane voltage traces of BOTH cells to look like, on the same time axis.

1. When Cell A is bursting, what should Cell B's voltage be doing?
2. What should happen right as Cell A's burst ends?
3. Roughly, should the two cells' bursts overlap, or be cleanly alternating?

*Your Answer Here (describe or sketch your predicted traces):*

### Step 2 — Build One HCO-Capable Cell
Each half-center needs a cell that can burst or adapt strongly — a plain fast-spiking cell (no adaptation) won't produce a clean rhythm.

> **Suggested prompt:** *"Write NEURON Python code for a single-compartment cell with Hodgkin-Huxley Na and K channels plus a slow potassium (M-type or Ca-dependent) current for adaptation/bursting, similar to what I built in Set 5. Give it a way to inject constant current and plot its voltage trace."*

Paste and run the code below. Confirm this single cell bursts or adapts clearly on its own before moving on.

In [ ]:
# Your single HCO-capable cell code here:


### Step 3 — Connect Two Cells with Reciprocal Inhibition
Now build a second, identical cell, and connect them: Cell A inhibits Cell B, and Cell B inhibits Cell A.

> **Suggested prompt:** *"I have this single-cell NEURON model: [paste your Step 2 code]. Create two instances of it, and connect them with reciprocal inhibitory (GABA-like) synapses — each cell's spikes should trigger an inhibitory postsynaptic conductance in the other cell."*

Paste and run the code below.

In [ ]:
# Your two-cell reciprocal-inhibition code here:


### Step 4 — Tune for Clean Alternation
Run it. You'll probably NOT get clean alternation on the first try — that's expected, and it's the actual point of this exercise.

> **Suggested prompt:** *"My two-cell reciprocally-inhibited network isn't alternating cleanly — [describe what you're actually seeing: e.g., 'both cells fire together', 'one cell dominates permanently', 'it's irregular']. Given that oscillation requires a slow escape mechanism, what parameters should I adjust — inhibitory synaptic weight, adaptation/bursting conductance strength, or current injection — and why?"*

Iterate. Paste your working code once you get clean, stable, alternating bursting.

In [ ]:
# Your tuned, working half-center oscillator code here:


### Step 5 — Verify Against Your Prediction
Compare your working traces to what you sketched in Step 1.

1. Did you correctly predict the alternation pattern?
2. What did you get wrong, and why — was it about the escape mechanism, the inhibition strength, or something else?
3. Real half-center oscillators (like the leech heartbeat generator) use a slow $I_h$ current for escape. Did your chosen mechanism (adaptation or bursting current) work the same way, or differently?

*Your Answer Here:*

### Capstone Reflection
This closes out three Sets of work: single cells (Set 5), synapses (Set 6), and network motifs (Set 7). In two or three sentences:

- What was the single hardest parameter to get right, and why?
- If you had to explain to someone who's only completed Set 4 why a half-center oscillator needs MORE than just two cells and an inhibitory connection, what would you say?

*Your Answer Here:*

**Congratulations — you've built, in one sitting, what used to be a half-semester modeling project.**

---
## Capstone Knowledge Bank — Half-Center Oscillator Design

1. What are the two essential ingredients of a half-center oscillator, beyond just "two cells and a synapse"?
2. Why would two plain fast-spiking cells (no adaptation, no bursting current) fail to produce a half-center rhythm?
3. Name one real biological example of a half-center oscillator, and the specific slow current it uses for escape.
4. If you increased inhibitory synaptic weight without changing anything else, would you expect the oscillation period to get longer or shorter? Why?
5. Why is 'predict first, then build' a more informative exercise than just building and seeing what happens?